# RV-ANDROID playground

# Config

In [1]:
# Log in HF

import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv(override=True)

hf_token = os.getenv('HF_TOKEN')

login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Local


In [2]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# %autoreload 0: Desativa o autoreload
# %autoreload 1: Recarrega apenas módulos importados usando %aimport
# %autoreload 2: Recarrega todos os módulos (exceto aqueles especificados)

%lsmagic

Available line magics:
%aimport  %alias  %alias_magic  %autoawait  %autocall  %automagic  %autoreload  %autosave  %bookmark  %cat  %cd  %clear  %code_wrap  %colors  %conda  %config  %connect_info  %cp  %debug  %dhist  %dirs  %doctest_mode  %ed  %edit  %env  %gui  %hist  %history  %killbgscripts  %ldir  %less  %lf  %lk  %ll  %load  %load_ext  %loadpy  %logoff  %logon  %logstart  %logstate  %logstop  %ls  %lsmagic  %lx  %macro  %magic  %mamba  %man  %matplotlib  %micromamba  %mkdir  %more  %mv  %notebook  %page  %pastebin  %pdb  %pdef  %pdoc  %pfile  %pinfo  %pinfo2  %pip  %popd  %pprint  %precision  %prun  %psearch  %psource  %pushd  %pwd  %pycat  %pylab  %qtconsole  %quickref  %recall  %rehashx  %reload_ext  %rep  %rerun  %reset  %reset_selective  %rm  %rmdir  %run  %save  %sc  %set_env  %store  %sx  %system  %tb  %time  %timeit  %unalias  %unload_ext  %uv  %who  %who_ls  %whos  %xdel  %xmode

Available cell magics:
%%!  %%HTML  %%SVG  %%bash  %%capture  %%code_wrap  %%debug  %%file  %

Requisites

`sudo apt install python3.12-dev nvidia-cuda-toolkit bitsandbytes triton`

```
nvidia-smi
nvcc --version
```

In [ ]:
# Check GPU

import torch
if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available")

## Google colab

In [ ]:
!pip install -q gradio diffusers transformers accelerate torch Pillow python-dotenv torchvision ollama
!pip install -q -U bitsandbytes

#datasets

In [ ]:
# Clone RVSec
from google.colab import userdata, drive

!rm -Rf sample_data/

#https://github.com/ad17171717/YouTube-Tutorials/blob/main/Google%20Colab%20Tutorials/Google_Colab_%2B_Git_Pushing_Changes_to_a_GitHub_Repo!.ipynb
!git config --global user.name "phtcosta"
!git config --global user.email "phtcosta@gmail.com"

# https://github.com/settings/tokens
github_token = userdata.get('GITHUB_TOKEN')
!git clone --branch develop https://{github_token}@github.com/PAMunb/rvsec.git

%cd rvsec/rv-android/
!pip install -q -r requirements.txt

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Log in HF
from huggingface_hub import login
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
# !git status
# !pwd

In [ ]:
## drive.flush_and_unmount()
## !git add --all
## !git commit -a -m "Just testing"
## !git remote -v

#  Experiments

In [3]:
# Imports

from IPython.display import Markdown, display, update_display #, Image
import gradio as gr
from PIL import Image
import numpy as np
import os
import glob
import json
import networkx as nx
import matplotlib.pyplot as plt
import gradio as gr
import io
from typing import List
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, AutoModelForSeq2SeqLM
import torch
from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor, WhisperConfig, WhisperForConditionalGeneration

from rvandroid.llm.huggingface_llm import HuggingFaceLLM
from rvandroid.llm.llm import LanguageModel
from rvandroid.llm.ollama_llm import OllamaLLM
# import rvandroid.parser.droidbot.droidbot_state_parser as state_parser
import rvandroid.parser.droidbot.droidbot_state_parser_novo as state_parser
# from rvandroid.parser.droidbot.state_parser import StateParser
from rvandroid.parser.static import gator_parser, reach_parser, gesda_parser, static_analysis_parser
from rvandroid.model.static import StaticAnalysisData
from rvandroid.model.classes import Classes
from rvandroid.model.window import Windows
from rvandroid.model.wtg import WindowTransitionGraph
from rvandroid.constants import *
from rvandroid.app import App
from rvandroid.llm.llm import LanguageModel
from rvandroid.llm.model_factory import ModelFactory
from rvandroid.llm.prompt_generator import PromptGenerator
from rvandroid.llm.prompt_strategy import PromptStrategyFactory
from rvandroid.llm.llm_config import LLMConfiguration
from rvandroid.service.llm_action_service import LLMActionService


/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/pydantic/_internal/_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'fields' has been removed
  warnings.warn(message, UserWarning)


In [ ]:
# Constants

# LLM_OLLAMA = "ollama"
# LLM_HF = "hf"


In [ ]:
# DEFAULT_LLM_TYPE = LLM_HF
# DEFAULT_MODEL = HuggingFaceLLM.LLAMA
# MODELS = HuggingFaceLLM.MODELS

# # DEFAULT_LLM_TYPE = LLM_OLLAMA
# # DEFAULT_MODEL = OllamaLLM.LLAMA
# # MODELS = OllamaLLM.MODELS


# def create_llm(model_name: str = DEFAULT_MODEL, type: str = DEFAULT_LLM_TYPE) -> LanguageModel:
#     if type == LLM_OLLAMA:
#         return OllamaLLM(model_name)
#     return HuggingFaceLLM(model_name)


In [4]:
def read_text_file(file_path):
    with open(file_path, 'r') as file:
        text = file.read()
    return text

def read_files_by_extension(folder: str, extension: str = "*.gesda"):
    files = glob.glob(os.path.join(folder, extension))
    for file in files:
      text = read_text_file(file)
      yield file, text


## Static Analysis

In [ ]:
static_folder = "/home/pedro/desenvolvimento/RV_ANDROID/teste_llm/static"
# static_folder = "/content/drive/MyDrive/llms/rvandroid/static"


def create_messages(system_msg: str, prompt: str, json_text: str) -> list[dict[str, str]]:
    messages=[
        {"role": "system", "content": system_msg },
        {"role": "user", "content": prompt.format(json_text)}
    ]
    return messages


### Parser

In [ ]:
# Individual parsers

apk = "cryptoapp.apk"
package = "br.unb.cic.cryptoapp"
windows = Windows()

reach_file = static_folder+"/cryptoapp.apk.reach"
classes = reach_parser.read_reachable_methods(reach_file)

gator_file = static_folder+"/cryptoapp.apk.wtg"
wtg = gator_parser.parse_gator_file(gator_file, package, classes, windows)

gesda_file = static_folder+"/cryptoapp.apk.gesda"
gesda_parser.parse_gesda_file(gesda_file, package, classes, windows)

print("\n\n...................................................")
for window in windows.windows:
    print(f"window={window}")
    for widget_id in window.widgets:
        widget = window.widgets[widget_id]
        print(f"\t-widget={widget}")
        for listener in widget.events:
            print(f"\t\t-listener={listener}")

# print("\n\n...................................................")
# print(f"wtg={wtg}")

In [ ]:
# All parsers

apk = "cryptoapp.apk"
package = "br.unb.cic.cryptoapp"

static_data = static_analysis_parser.read_static_analysis_files(static_folder, apk, package)

print("\n\n...................................................")
for window in static_data.windows.windows:
    print(f"window={window}")
    for widget_id in window.widgets:
        widget = window.widgets[widget_id]
        print(f"\t-widget={widget}")
        for listener in widget.events:
            print(f"\t\t-listener={listener}")

print("\n\n...................................................")
print(f"wtg={static_data.wtg}")

In [ ]:
# Gradio

from rvandroid.app import App


def get_apks_files(folder: str):
    files = []
    for file in os.listdir(folder):
        if file.endswith(".apk"):
            files.append(file)
    return files

def plot_graph(G):
    """
    Function that plots the graph and returns the image
    """
    # Clear current figure and create a new one
    plt.clf()
    fig = plt.figure(figsize=(10, 8))

    # Create graph layout
    pos = nx.spring_layout(G)

    # Draw the graph
    nx.draw(G, pos, with_labels=True, node_color='lightblue',
            node_size=500, arrowsize=20, font_size=10,
            font_weight='bold', arrows=True)

    # Convert plot to numpy array
    fig.canvas.draw()

    # Get the RGBA buffer from the figure
    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    buf.shape = (h, w, 4)

    # Convert RGBA to RGB
    img_array = buf[:, :, :3]

    # Close the figure to free memory
    plt.close()

    return img_array


files = get_apks_files(static_folder)
# print(len(files))
current_index = 0
static_data: StaticAnalysisData

def parse_file(file):
    apk_file = os.path.join(static_folder, file)
    app = App(apk_file)
    package = app.package_name
    print(f"Parsing file: {apk_file}")
    global static_data
    static_data = static_analysis_parser.read_static_analysis_files(static_folder, file, package)
    return static_data.classes.to_json(), static_data.windows.to_json(), static_data.wtg.to_json(), plot_graph(static_data.wtg.graph)

def display_file(index):
    return files[index]

def advance_file():
    """Advances to the next file in the list."""
    global current_index
    current_index = (current_index + 1) % len(files)  # Wraps around to the beginning if at the end
    return display_file(current_index)

def go_back_file():
    """Goes back to the previous file in the list."""
    global current_index
    current_index = (current_index - 1) % len(files)  # Wraps around to the end if at the beginning
    return display_file(current_index)

with gr.Blocks() as demo:
    with gr.Row():
        filename = gr.Textbox(label="APK", lines=1, value=display_file(current_index))

    with gr.Row():
        previous_button = gr.Button("Previous")
        next_button = gr.Button("Next")

    with gr.Row():
        parse_button = gr.Button("Parse")

    # with gr.Row():
    with gr.Tabs():
        with gr.TabItem("Classes"):
            classes_json = gr.JSON()

        with gr.TabItem("Windows"):
            windows_json = gr.JSON()

        with gr.TabItem("WTG"):
            wtg_json = gr.JSON()

        with gr.TabItem("WTG (img)"):
            wtg_img = gr.Image()

    previous_button.click(go_back_file, outputs=filename)
    next_button.click(advance_file, outputs=filename)
    parse_button.click(parse_file, inputs=filename, outputs=[classes_json, windows_json, wtg_json, wtg_img])

demo.launch(debug=True)

### GESDA

In [ ]:
base_gesda_system_msg = """You are an expert assistant in testing the interface of Android applications, and you use this knowledge to make useful summaries about the components (activities, windows, widgets) contained on the screen. Some widgets may have information about which method will be called when it is clicked, others may have information about the assignment of this widget to a field declared in the class, listing all those that are relevant in the context of interface testing, and suggesting the possible actions on this component (click, set text, select item). The information about the application that must be understood is contained in a string in json format, which will be passed to you.
"""
base_gesda_prompt = "Make a summary of the application 'cryptoapp' which has the following information in json format: {}"



In [ ]:
# Basic example (GESDA)

text = read_text_file(static_folder+"/cryptoapp.apk.gesda")
print(text)
messages=create_messages(base_gesda_system_msg, base_gesda_prompt, text)
print(messages)

print("Generating ...")
llm = create_llm()
response_gesda = llm.generate(messages)
display(Markdown(response_gesda))
llm.clean()

del llm

In [ ]:
# # Basic Example - LOCAL (parser)

# text = read_text_file(static_folder+"/cryptoapp.apk.gesda")

# # fazer o parse do arquivo .gesda (json) para dictionary
# def read_gesda_file(file_path: str):
#     try:
#         with open(file_path, 'r') as file:
#             data = json.load(file)
#         return data
#     except FileNotFoundError:
#         print(f"File not found: {file_path}")
#         return None
#     except json.JSONDecodeError:
#         print(f"Failed to parse JSON from file: {file_path}")
#         return None

# gesda = read_gesda_file(static_folder+"/cryptoapp.apk.gesda")
# print(gesda)

# def parse_gesda_file(gesda_data):
#     for window in gesda_data["windows"]:
#         print(window["name"])



# parse_gesda_file(gesda)


In [ ]:
# Gradio

llm = create_llm()

# Get GESDA files (path) and texts (contents)
files, texts = get_gesda_files(static_folder)

current_index = 0  # Index of the currently displayed file
selected_model = DEFAULT_MODEL # Currently selected model

def get_system_prompt():
   return base_gesda_system_msg

def get_user_prompt():
   return base_gesda_prompt

def generate_output(system_prompt, user_prompt):
  messages = create_messages(system_prompt, user_prompt, texts[current_index])
  response = llm.generate(messages)
  return response

def display_file(index):
    """Returns the filename at the given index."""
    return files[index]

def process_selection(selected_option):
  """Processes the model selection."""
  global selected_model
  selected_model = selected_option
  print(f"You selected: {selected_option}")
  global llm # Make llm global so it can be reassigned
  llm = create_llm(selected_model, DEFAULT_LLM_TYPE) # Initialize the LLM with the selected model

def advance_file():
    """Advances to the next file in the list."""
    global current_index
    current_index = (current_index + 1) % len(files)  # Wraps around to the beginning if at the end
    return display_file(current_index)

def go_back_file():
    """Goes back to the previous file in the list."""
    global current_index
    current_index = (current_index - 1) % len(files)  # Wraps around to the end if at the beginning
    return display_file(current_index)

def reset(system_prompt, user_prompt, result_text):
  """Resets the system and user prompts and the result text."""
  return base_gesda_system_msg, base_gesda_prompt, ""

def clear_memory():
  """Clears the LLM memory and CUDA cache."""
  if llm is not None:
    llm.clean()
  torch.cuda.empty_cache()
  return ""

with gr.Blocks() as demo:
    with gr.Row():
      filename = gr.Textbox(label="GESDA file", lines=1, value=display_file(current_index))

    with gr.Row():
      previous_button = gr.Button("Previous")
      next_button = gr.Button("Next")

    with gr.Row():
      system_textbox = gr.Textbox(label="System Prompt", value=get_system_prompt()) #, lines=5)
      prompt_textbox = gr.Textbox(label="User Prompt", value=get_user_prompt()) #, lines=3)

    with gr.Row():
      model_dropdown = gr.Dropdown(
        label="Select MODEL",
        choices=llm.models(),
        value=DEFAULT_MODEL
      )
      with gr.Row():
        generate_button = gr.Button("Generate")
        reset_button = gr.Button("Reset")

    with gr.Row():
      result_textbox = gr.Textbox(lines=10)

    with gr.Row():
      clear_button = gr.Button("Clear memory")

    previous_button.click(go_back_file, outputs=filename)
    next_button.click(advance_file, outputs=filename)
    model_dropdown.change(fn=process_selection, inputs=model_dropdown)
    generate_button.click(generate_output, inputs=[system_textbox, prompt_textbox], outputs=result_textbox)
    reset_button.click(reset, inputs=[system_textbox, prompt_textbox, result_textbox], outputs=[system_textbox, prompt_textbox, result_textbox])
    clear_button.click(clear_memory)

demo.launch(debug=True)

In [ ]:
# Clean up

if llm is not None:
    llm.clean()

del llm
del demo

### GATOR

In [ ]:
base_gator_system_msg = """You are an expert in testing the interface of Android applications, and you use this knowledge to make useful summaries about the components (activities, windows, widgets and transitions between windows) contained on the screen. Some widgets may have information about which method will be called when clicked, others may have information about the assignment of this widget to a field declared in the class. List all those that are relevant in the context of interface testing, and suggest the possible actions on this component (click, set text, select item). The main objective is to increase the test coverage according to the window transition graph to try to cover all the activities.
There are two basic sections in the json: one with information about the application windows and another with information about the transitions between windows, including the events that cause this transition. The information about the windows is in the following format "{"id":1349,"name":"br.unb.cic.cryptoapp.MainActivity"}", where the activity br.unb.cic.cryptoapp.MainActivity has the identifier 1349. Another example: "{ "id": 1336, "name": "br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity" }, where the activity br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity has the identifier 1336"
The information about the transitions between the windows is in the following format:
{
"sourceId": 1349,
"getTargetId": 1336,
"events": [
{
"type": "click",
"handler": "\u003cbr.unb.cic.cryptoapp.MainActivity: void showScreenMessageDigest(android.view.View)\u003e",
"widgetId": 2131296357,
"widgetClass": "android.widget.Button",
"widgetName": "buttonMessageDigest"
}
]
}
The above example shows a transition between the activities (windows) br.unb.cic.cryptoapp.MainActivity (with id 1349) and br.unb.cic.cryptoapp.messagedigest.MessageDigestActivity (with id 1336). This transition is activated when the button 'buttonMessageDigest' is clicked. When the button is clicked, the event is passed to the method "\u003cbr.unb.cic.cryptoapp.MainActivity: void showScreenMessageDigest(android.view.View)\u003e", which will start the intent corresponding to MessageDigestActivity
"""
base_gator_prompt = "Create a document containing information about the 'cryptoapp' application. The response should contain the list of all windows and a mapping of the transitions between them, including the event that caused the transition between screens. The text containing the application data: {}"

In [ ]:
# Basic example (GATOR)

text = read_text_file(static_folder+"/cryptoapp.apk.wtg")
print(text)
# messages=create_messages(base_gesda_system_msg, base_gator_prompt, text)
messages=create_messages(base_gator_system_msg, base_gator_prompt, text)
print(messages)

print("Generating ...")
llm = create_llm()
response_gator = llm.generate(messages)
display(Markdown(response_gator))
llm.clean()

del llm

### Gradio

In [ ]:
def get_files_by_extension(folder: str, extension = "*.gesda"):
    filenames = []
    texts = []
    for file, text in read_files_by_extension(folder, extension):
      filenames.append(file)
      texts.append(text)
    return filenames, texts


# GESDA
# files, texts = get_files_by_extension(static_folder, "*.gesda")
# gradio_system_prompt = base_gesda_system_msg
# gradio_user_prompt = base_gesda_prompt

# GATOR
files, texts = get_files_by_extension(static_folder, "*.wtg")
gradio_system_prompt = base_gator_system_msg
gradio_user_prompt = base_gator_prompt



llm = create_llm()

current_index = 0  # Index of the currently displayed file
selected_model = DEFAULT_MODEL # Currently selected model

def get_system_prompt():
   return gradio_system_prompt

def get_user_prompt():
   return gradio_user_prompt

def generate_output(system_prompt, user_prompt):
  messages = create_messages(system_prompt, user_prompt, texts[current_index])
  response = llm.generate(messages)
  return response

def display_file(index):
    """Returns the filename at the given index."""
    return files[index]

def process_selection(selected_option):
  """Processes the model selection."""
  global selected_model
  selected_model = selected_option
  print(f"You selected: {selected_option}")
  global llm # Make llm global so it can be reassigned
  llm = create_llm(selected_model, DEFAULT_LLM_TYPE) # Initialize the LLM with the selected model

def advance_file():
    """Advances to the next file in the list."""
    global current_index
    current_index = (current_index + 1) % len(files)  # Wraps around to the beginning if at the end
    return display_file(current_index)

def go_back_file():
    """Goes back to the previous file in the list."""
    global current_index
    current_index = (current_index - 1) % len(files)  # Wraps around to the end if at the beginning
    return display_file(current_index)

def reset(system_prompt, user_prompt, result_text):
  """Resets the system and user prompts and the result text."""
  return gradio_system_prompt, gradio_user_prompt, ""

def clear_memory():
  """Clears the LLM memory and CUDA cache."""
  if llm is not None:
    llm.clean()
  torch.cuda.empty_cache()
  return ""

with gr.Blocks() as demo:
    with gr.Row():
      filename = gr.Textbox(label="File", lines=1, value=display_file(current_index))

    with gr.Row():
      previous_button = gr.Button("Previous")
      next_button = gr.Button("Next")

    with gr.Row():
      system_textbox = gr.Textbox(label="System Prompt", value=get_system_prompt()) #, lines=5)
      prompt_textbox = gr.Textbox(label="User Prompt", value=get_user_prompt()) #, lines=3)

    with gr.Row():
      model_dropdown = gr.Dropdown(
        label="Select MODEL",
        choices=llm.models(),
        value=DEFAULT_MODEL
      )
      with gr.Row():
        generate_button = gr.Button("Generate")
        reset_button = gr.Button("Reset")

    with gr.Row():
      result_textbox = gr.Textbox(lines=10)

    with gr.Row():
      clear_button = gr.Button("Clear memory")

    previous_button.click(go_back_file, outputs=filename)
    next_button.click(advance_file, outputs=filename)
    model_dropdown.change(fn=process_selection, inputs=model_dropdown)
    generate_button.click(generate_output, inputs=[system_textbox, prompt_textbox], outputs=result_textbox)
    reset_button.click(reset, inputs=[system_textbox, prompt_textbox, result_textbox], outputs=[system_textbox, prompt_textbox, result_textbox])
    clear_button.click(clear_memory)

demo.launch(debug=True)

In [ ]:
# Clean up

if llm is not None:
    llm.clean()

del llm
del demo

## Screen to Text

In [6]:
screenshots_folder = "/home/pedro/desenvolvimento/RV_ANDROID/teste_llm/screenshots"
# screenshots_folder = "/content/drive/MyDrive/llms/rvandroid/screenshots"

def read_screens_info(folder_path: str):
  files = os.listdir(folder_path)
  for file in files:
    if file.endswith(".png"):
      png_file = os.path.join(folder_path, file)
      state_file = os.path.join(folder_path, file.replace(".png", ".state"))
      yield png_file, state_file

def read_screens_info_with_static(folder_path: str):
  files = os.listdir(folder_path)
  for file in files:
    if file.endswith(".png"):
      png_file = os.path.join(folder_path, file)
      state_file = os.path.join(folder_path, file.replace(".png", ".state"))
      reach_file = os.path.join(folder_path, file.replace(".png", ".reach"))
      gesda_file = os.path.join(folder_path, file.replace(".png", ".gesda"))
      gator_file = os.path.join(folder_path, file.replace(".png", ".wtg"))
      yield png_file, state_file, reach_file, gesda_file, gator_file

def read_droidbot_state(filename):
    with open(filename, 'r') as file:
        return json.load(file)

def read_app_names():
  selected_apps = []
  files = os.listdir(screenshots_folder)
  for file in files:
    file_path = os.path.join(screenshots_folder, file)
    if os.path.isdir(file_path):
      selected_apps.append({"name": file, "path": file_path})
  return selected_apps

### droidbot-GPT

 - a view with text "Crypto App";
 - a view with text "Message Digest";
 - a view that can click (0), scroll up (1), scroll down (2);
 - a view with text "Select" that can click (3);
 - a editable view with text "Input text ..." that can edit (4), click (5);
 - a view with text "GENERATE HASH" that can click (6);
 - a key to go back (7)

### rv-android

In [ ]:
apk = "cryptoapp.apk"
app_folder = screenshots_folder+"/"+apk
info_file = app_folder+"/001.state"
screen_info = read_droidbot_state(info_file)

app = App(os.path.join(app_folder, apk))
package = app.package_name

static_data = static_analysis_parser.read_static_analysis_files(app_folder, apk, package)

screen_description = state_parser.parse(screen_info, static_data)

print(screen_description)

In [ ]:

lista = []
for app in read_app_names():
    print(app["name"])
    for png_file, state_file, reach_file, gesda_file, gator_file  in read_screens_info_with_static(app["path"]):
        lista.append({"state": state_file, "reach": reach_file, "gesda": gesda_file, "gator": gator_file, "image": png_file})

# for png_file, state_file, reach_file, gesda_file, gator_file  in read_screens_info_with_static(os.path.abspath(screenshots_folder)):
#     lista.append({"state": state_file, "reach": reach_file, "gesda": gesda_file, "gator": gator_file, "image": png_file})

for item in lista:
    print(item)
print(len(lista))

indice_atual = 0

def mostrar_imagem(indice):
    """Mostra a imagem atual da lista."""
    try:
        imagem = Image.open(lista[indice]["image"])
        return imagem
    except FileNotFoundError:
        return "Imagem não encontrada"

def descrever_tela():
    print(f"alterando texto do indice: {indice_atual}")
    arquivo_state = lista[indice_atual]["state"]
    reach_file = lista[indice_atual]["reach"]
    gesda_file = lista[indice_atual]["gesda"]
    gator_file = lista[indice_atual]["gator"]
    apk_name = os.path.basename(os.path.dirname(arquivo_state))
    apk_path = os.path.join(os.path.dirname(arquivo_state), apk_name)
    app = App(apk_path)
    screen_info = read_droidbot_state(arquivo_state)
    static_data = static_analysis_parser.parse(reach_file, gator_file, gesda_file, app.package_name)
    screen_description = state_parser.parse(screen_info, static_data)
    return screen_description.description

def avancar_imagem():
    """Avança para a próxima imagem da lista."""
    global indice_atual
    indice_atual = (indice_atual + 1) % len(lista)  # Volta ao início se chegar ao fim da lista
    return mostrar_imagem(indice_atual), lista[indice_atual]["state"]

def voltar_imagem():
    """Volta para a imagem anterior da lista."""
    global indice_atual
    indice_atual = (indice_atual - 1) % len(lista)  # Volta para o fim se chegar ao início da lista
    return mostrar_imagem(indice_atual), lista[indice_atual]["state"]

with gr.Blocks(theme=gr.themes.Base(),
               css=".white-bg-black-text {background-color: white !important; color: black !important;}"
               ) as demo:
    with gr.Row():
        arquivo = gr.Textbox(value=lista[indice_atual]["state"], show_label=False)

    with gr.Row():
        imagem = gr.Image(value=mostrar_imagem(indice_atual), type="pil", height=500)
        texto = gr.Markdown() #elem_classes="white-bg-black-text")

    with gr.Row():
        botao_anterior = gr.Button("Previous")
        botao_proximo = gr.Button("Next")
        botao_alterar = gr.Button("Descrever Tela")

    botao_anterior.click(voltar_imagem, outputs=[imagem, arquivo])
    botao_proximo.click(avancar_imagem, outputs=[imagem, arquivo])
    botao_alterar.click(descrever_tela, outputs=texto)

demo.launch()

### VQA

In [ ]:
def redimensionar_imagem(caminho_imagem, largura_maxima, altura_maxima):
    """
    Redimensiona uma imagem para as dimensões máximas especificadas,
    mantendo a proporção original.

    Args:
        caminho_imagem: O caminho para a imagem original.
        largura_maxima: A largura máxima desejada.
        altura_maxima: A altura máxima desejada.

    Returns:
        Uma imagem PIL redimensionada.
    """
    image = Image.open(caminho_imagem)
    image.thumbnail((largura_maxima, altura_maxima))
    return image

In [ ]:
# facebook/blip-large: Este modelo é um dos mais populares e oferece um bom equilíbrio entre desempenho e tamanho. Ele é capaz de responder a perguntas complexas sobre imagens e gerar descrições detalhadas.
# google/flan-t5-xxl: Embora seja um modelo maior, o Flan-T5-XXL pode ser usado para VQA com bom desempenho em GPUs T4, especialmente se você otimizar o uso da memória. Ele é conhecido por sua capacidade de gerar texto de alta qualidade.
# Salesforce/blip-2-flan-t5-xl: Este modelo combina o poder do BLIP-2 para visão com o modelo Flan-T5-XL para linguagem, oferecendo resultados impressionantes em tarefas de VQA.

# Escolha um modelo
# default: dandelin/vilt-b32-finetuned-vqa
# model_name = "facebook/blip-large"
# model_name = "Salesforce/blip-2-flan-t5-xl"
# model_name = "google/flan-t5-xxl"
model_name = "google/flan-t5-small"

# Crie o pipeline de VQA
vqa_pipeline = pipeline("visual-question-answering", model=model_name)

# Carregue a imagem
image_path = screenshots_folder+"/001.png"
# image_path = "/content/drive/MyDrive/llms/cryptoapp/001.png"
image = Image.open(image_path)
image = redimensionar_imagem(image_path, 512, 512)

# Defina a pergunta
# question = "O que está acontecendo na imagem?"
question = """
Instruções
Descreva a tela do aplicativo Android em detalhes, conforme as instruções fornecidas.

Formato de Resposta
A resposta deve ser estruturada em um formato de tabela ou lista, facilitando a identificação e o uso das informações para testes.

Considerações Adicionais
Adapte este prompt para suas necessidades específicas, incluindo detalhes sobre o aplicativo e os tipos de teste que você deseja realizar.
Seja claro e específico nas suas instruções para obter uma resposta mais precisa e útil.
Use a criatividade para explorar diferentes tipos de interações e ações que podem ser realizadas na tela.
"""

# Obtenha a resposta
result = vqa_pipeline(image, question)
print(result)

In [ ]:
torch.cuda.empty_cache()
del vqa_pipeline

In [ ]:
torch.cuda.empty_cache()

In [ ]:

image_path = screenshots_folder+"/002.png"

LLAVA_0_5B = "llava-hf/llava-interleave-qwen-0.5b-hf"
LLAVA_7B = "llava-hf/llava-interleave-qwen-7b-hf"

pipe = pipeline("image-text-to-text", model=LLAVA_0_5B)

messages = [
     {
         "role": "user",
         "content": [
             {
                 "type": "image",
                 "image": image_path,
             },
             {"type": "text", "text": "describe in detail the following screenshot of an android application. Identify the components (buttons, fields, spinners, etc.) that are clickable, editable or selectable, indicating possible actions on them"},
         ],
     }
 ]





In [ ]:
outputs = pipe(text=messages, max_new_tokens=300, return_full_text=True)

outputs[0]["generated_text"]

#'The screenshot displays a mobile application interface with a blue background and white text. At the top, there is a section titled "Crypto App" with a date and time stamp of 11:54. Below this, there is a section titled "Message Digest" with a dropdown menu that allows the user to select a message to digest. The selected message is highlighted with a red background.\n\nOn the right side of the screen, there is a section titled "Generate Hash" with a blue button that says "Generate Hash" and a spinning spinner icon. The spinner icon indicates that the user can generate a hash of the selected message.\n\nOn the bottom left of the screen, there is a section titled "Select Text" with a blue field that allows the user to select a text to insert into the digest. The field is highlighted with a red background.\n\nThe overall layout of the application is simple and user-friendly, with clear and concise text and icons. The design is clean and modern, with a focus on simplicity and ease of use.'}]

In [ ]:
del pipe

In [ ]:
!pip install -q transformers torch torchvision Pillow opencv-python pytesseract
!sudo apt install -y tesseract-ocr
!sudo apt install -y libtesseract-dev

from transformers import ViTFeatureExtractor, ViTModel, BertTokenizer, BertModel
from PIL import Image
import torch
import cv2
import pytesseract

# Modelos de visão
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224')
model_vision = ViTModel.from_pretrained('google/vit-base-patch16-224')

# Modelos de linguagem
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model_language = BertModel.from_pretrained('bert-base-uncased')

# Configuração do Tesseract OCR
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'  # Ajuste o caminho para o seu Tesseract



def identificar_elementos(imagem):
  """Identifica elementos interativos na imagem usando OpenCV."""
  # Converta a imagem para escala de cinza
  gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)

  # Use detecção de bordas para encontrar contornos
  edges = cv2.Canny(gray, 50, 150, apertureSize=3)

  # Encontre contornos
  contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

  elementos = []
  for contour in contours:
    # Obtenha as coordenadas do retângulo delimitador
    x, y, w, h = cv2.boundingRect(contour)

    # Considere apenas contornos com área razoável
    if w * h > 1000:
      elementos.append({'x': x, 'y': y, 'w': w, 'h': h})

  return elementos

def extrair_texto(imagem, elemento):
  """Extrai texto de um elemento usando Tesseract OCR."""
  x, y, w, h = elemento['x'], elemento['y'], elemento['w'], elemento['h']
  crop = imagem[y:y+h, x:x+w]
  texto = pytesseract.image_to_string(crop)
  return texto.strip()


# Carregue o screenshot
imagem = cv2.imread("/content/drive/MyDrive/llms/cryptoapp/001.png")

# Identifique os elementos interativos
elementos = identificar_elementos(imagem)

# Extraia o texto dos elementos
for elemento in elementos:
  elemento['texto'] = extrair_texto(imagem, elemento)

# Gere a descrição textual
descricao = "Tela com os seguintes elementos:\n"
for elemento in elementos:
  descricao += f"- {elemento['texto']} ({elemento['x']}, {elemento['y']}, {elemento['w']}, {elemento['h']})\n"

# Gere as possíveis ações
acoes = []
for elemento in elementos:
  if elemento['texto']:
    acoes.append(f"Interagir com o elemento: {elemento['texto']}")

print(descricao)
print(acoes)

## Prompt Generator

In [7]:
screenshots_folder = "/home/pedro/desenvolvimento/RV_ANDROID/teste_llm/screenshots"
# screenshots_folder = "/content/drive/MyDrive/llms/rvandroid/screenshots"

apk = "cryptoapp.apk"
app_folder = screenshots_folder+"/"+apk
info_file = app_folder+"/001.state"

app = App(os.path.join(app_folder, apk))
package = app.package_name

screen_info = read_droidbot_state(info_file)

static_data = static_analysis_parser.read_static_analysis_files(app_folder, apk, package)

from rvandroid.llm.prompt_strategy_basic_001 import BasicPromptStrategy001
PromptStrategyFactory.register_strategy("basic", BasicPromptStrategy001)

strategy_type = "basic"
prompt_strategy = PromptStrategyFactory.create(strategy_type, static_data)

system_prompt = prompt_strategy.generate_system_prompt()
user_prompt = prompt_strategy.generate_user_prompt(screen_info)

print(f"System prompt:\n{system_prompt}")
print(f"User prompt:\n{user_prompt}")


Requested API level 33 is larger than maximum we have, returning API level 28 instead.


$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
view: {'package': 'br.unb.cic.cryptoapp', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[42, 101], [316, 172]], 'resource_id': None, 'checked': False, 'text': 'Crypto App', 'class': 'android.widget.TextView', 'scrollable': False, 'selected': False, 'long_clickable': False, 'parent': 5, 'temp_id': 6, 'size': '274*71'}
view: {'package': 'br.unb.cic.cryptoapp', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': True, 'is_password': False, 'focusable': True, 'enabled': True, 'content_description': 'More options', 'children': [], 'focused': False, 'bounds': [[975, 73], [1080, 199]], 'resource_id': None, 'checked': False, 'text': None, 'class': 'android.widget.ImageView', 'scrollable': False, 'selected': False, 'long_clic

System prompt:
You are an Android UI testing expert. Your task is to analyze the current app state and suggest the most effective testing actions.

Focus on:
1. Maximizing code coverage by targeting untested UI elements
2. Exercising important methods that directly or indirectly affect operations of interest, defined in formal specifications
3. Systematically exploring application states
4. Testing complex UI interactions and edge cases

For each action, provide:
- Action type (click, long_click, scroll, set_text, key_event)
- Target widget identifier or coordinates
- Parameters where needed (text input, scroll direction, etc.)
- Brief explanation of why you chose this action

Format your response as a valid JSON array of actions following this schema:
[
  {
    "action_type": "click",  
    "target": "widget_id_or_index",
    "params": {},  
    "explanation": "Brief explanation"
  },
  ...
]

IMPORTANT CONSTRAINTS ON ACTION SELECTION:
1. When faced with multiple possible paths (e.g.,

In [ ]:
apk = "cryptoapp.apk"
app_folder = screenshots_folder+"/"+apk
info_file = app_folder+"/001.state"

app = App(os.path.join(app_folder, apk))
package = app.package_name

screen_info = read_droidbot_state(info_file)

static_data = static_analysis_parser.read_static_analysis_files(app_folder, apk, package)

from rvandroid.llm.prompt_strategy_basic_001 import BasicPromptStrategy001
PromptStrategyFactory.register_strategy("basic", BasicPromptStrategy001)

service = LLMActionService(static_data) #, "huggingface", "Qwen/Qwen2.5-3B-Instruct", "basic")
result = service.process_state(screen_info)
print(result)

del service

In [ ]:
print(result)

In [ ]:
apk = "cryptoapp.apk"
app_folder = screenshots_folder+"/"+apk
info_file = app_folder+"/001.state"

app = App(os.path.join(app_folder, apk))
package = app.package_name

screen_info = read_droidbot_state(info_file)

static_data = static_analysis_parser.read_static_analysis_files(app_folder, apk, package)

config = LLMConfiguration(model_type="huggingface", model_name="Qwen/Qwen2.5-3B-Instruct", strategy_type="basic")

service = LLMActionService(static_data, config=config)
result = service.process_state(screen_info)
print(result)

del service


In [ ]:

apk = "cryptoapp.apk"
app_folder = screenshots_folder+"/"+apk
info_file = app_folder+"/001.state"

app = App(os.path.join(app_folder, apk))
package = app.package_name

screen_info = read_droidbot_state(info_file)

static_data = static_analysis_parser.read_static_analysis_files(app_folder, apk, package)

from rvandroid.llm.prompt_strategy_basic_001 import BasicPromptStrategy001
prompt_strategy = BasicPromptStrategy001(static_data)

system_prompt = prompt_strategy.generate_system_prompt()
user_prompt = prompt_strategy.generate_user_prompt(screen_info)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

llm = ModelFactory.create("huggingface", "Qwen/Qwen2.5-3B-Instruct")

result = llm.generate(messages)
print(f"result:\n{result}")

llm.clean()
del llm

In [ ]:
# Gradio
def get_model_types():
    """
    Returns a list of available model types.

    Returns:
        List of model types
    """
    return list(ModelFactory.get_available_types().keys())

def get_model_names(model_type: str):
    """
    Returns a list of available models for a specific type.
    Args:
        model_type: Type of model to get models for

    Returns:
        List of model names
    """
    return ModelFactory.get_available_models(model_type)[model_type]


def list_apks_folders() -> list[str]:
    apks = []
    for root, dirs, files in os.walk(screenshots_folder):
        for dir_name in dirs:
            if dir_name.endswith(".apk"):
                apks.append(dir_name)
    return sorted(apks)

def create_list_for_apk(apk_name):
    lista = []
    for png_file, state_file, reach_file, gesda_file, gator_file  in read_screens_info_with_static(os.path.abspath(screenshots_folder+"/"+apk_name)):
        lista.append({"state": state_file, "reach": reach_file, "gesda": gesda_file, "gator": gator_file, "image": png_file})
    return lista

model_type = "huggingface"
model_name = ""
llm = None
prompt_strategy = None
static_data = None
indice_atual = 0
apks = list_apks_folders()
lista = create_list_for_apk(apks[0])

from rvandroid.llm.prompt_strategy_basic_001 import BasicPromptStrategy001
# PromptStrategyFactory.register_strategy("basic", BasicPromptStrategy001)

def inicializar_modelo(name=HuggingFaceLLM.QWEN):
    """Inicializa o modelo apenas uma vez"""
    global llm
    if llm is None:
        global model_type, model_name
        if model_name:
            llm = ModelFactory.create(model_type, model_name)
        llm = HuggingFaceLLM(
            model_name=model_name,  # Escolha seu modelo
            device="cuda"
            # kv_cache=True,   # Ative o KV-cache para mensagens sequenciais
            # warmup=True      # Pré-carregue o modelo para eliminar latência inicial
        )
        # Pré-aquecer o modelo com uma geração simples
        llm.generate([{"role": "user", "content": "Hello"}], max_new_tokens=5)
    return "Modelo inicializado com sucesso!"

# inicializar_modelo()


def apk_changed(apk_name):
    print(f"APK selecionado: {apk_name}")
    
    global lista, indice_atual
    lista = create_list_for_apk(apk_name)
    indice_atual = 0
    
    return lista[indice_atual]["state"]

def mostrar_imagem(indice):
    try:
        imagem = Image.open(lista[indice]["image"])
        return imagem
    except FileNotFoundError:
        return "Image not found"
    
def generate_prompt():
    print(f"alterando texto do indice: {indice_atual}")
    arquivo_state = lista[indice_atual]["state"]
    reach_file = lista[indice_atual]["reach"]
    gesda_file = lista[indice_atual]["gesda"]
    gator_file = lista[indice_atual]["gator"]
    apk_name = os.path.basename(os.path.dirname(arquivo_state))
    apk_path = os.path.join(os.path.dirname(arquivo_state), apk_name)
    app = App(apk_path)
    screen_info = read_droidbot_state(arquivo_state)
    global prompt_strategy, static_data
    static_data = static_analysis_parser.parse(reach_file, gator_file, gesda_file, app.package_name)
    prompt_strategy = BasicPromptStrategy001(static_data)
    system_prompt = prompt_strategy.generate_system_prompt() 
    user_prompt = prompt_strategy.generate_user_prompt(screen_info)
    return user_prompt

def executar_prompt():
    print("Executando prompt...static_data: {static_data} ... prompt_strategy: {prompt_strategy}")
    global prompt_strategy, static_data
    if prompt_strategy and static_data:
        arquivo_state = lista[indice_atual]["state"]
        screen_info = read_droidbot_state(arquivo_state)
        
        # service = LLMActionService(static_data, 
        #                            model_type="huggingface", 
        #                            model_name="Qwen/Qwen2.5-3B-Instruct", 
        #                            prompt_strategy=prompt_strategy)        
        # return service.process_state(screen_info)
        
        global llm
        if llm is None:
            inicializar_modelo()
        return llm.generate(prompt_strategy.generate_prompts(screen_info))
    return ""

def avancar_imagem():
    """Avança para a próxima imagem da lista."""
    global indice_atual
    indice_atual = (indice_atual + 1) % len(lista)  # Volta ao início se chegar ao fim da lista
    return mostrar_imagem(indice_atual), lista[indice_atual]["state"]

def voltar_imagem():
    """Volta para a imagem anterior da lista."""
    global indice_atual
    indice_atual = (indice_atual - 1) % len(lista)  # Volta para o fim se chegar ao início da lista
    return mostrar_imagem(indice_atual), lista[indice_atual]["state"]

def atualizar_opcoes(model_type):
    return get_model_names(model_type)

with gr.Blocks(theme=gr.themes.Base()) as demo:
    with gr.Row():
        apks_combo = gr.Dropdown(choices=apks, label="APKs", interactive=True)
    with gr.Row():
        arquivo = gr.Textbox(value=lista[indice_atual]["state"], show_label=False)
    with gr.Row():
        botao_anterior = gr.Button("Previous")
        botao_proximo = gr.Button("Next")
        botao_gerar_prompt = gr.Button("Generate Prompt")        
    with gr.Row():
        imagem = gr.Image(value=mostrar_imagem(indice_atual), type="pil", height=500)
        texto = gr.Markdown()
    with gr.Row():
        types_combo = gr.Dropdown(choices=get_model_types(), label="Types", interactive=True, value=model_type)
        model_combo = gr.Dropdown(choices=get_model_names(model_type), label="Models", interactive=True)
    with gr.Row():
        botao_executar_prompt = gr.Button("Execute Prompt")
    with gr.Row():
        resultado_execucao = gr.Textbox(value="", show_label=False)

    apks_combo.change(fn=apk_changed, inputs=apks_combo, outputs=[imagem, arquivo])
    botao_anterior.click(voltar_imagem, outputs=[imagem, arquivo])
    botao_proximo.click(avancar_imagem, outputs=[imagem, arquivo])
    botao_gerar_prompt.click(generate_prompt, outputs=texto)
    types_combo.change(fn=atualizar_opcoes, inputs=types_combo, outputs=model_combo)
    botao_executar_prompt.click(executar_prompt, outputs=resultado_execucao)
    
demo.launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


alterando texto do indice: 0
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
view: {'package': 'byrne.utilities.hashpass', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[42, 101], [281, 172]], 'resource_id': None, 'checked': False, 'text': 'HashPass', 'class': 'android.widget.TextView', 'scrollable': False, 'selected': False, 'long_clickable': False, 'parent': 3, 'temp_id': 4, 'size': '239*71'}
view: {'package': 'byrne.utilities.hashpass', 'visible': False, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[1080, 63], [1080, 210]], 'resource_id': None, 'checked': False, 'text': None, 'class': 'android.widget.LinearLayout', 'scrollable': Fals

Traceback (most recent call last):
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/blocks.py", line 2098, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspac

alterando texto do indice: 1


$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
view: {'package': 'com.gianlu.dnshero', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': True, 'is_password': False, 'focusable': True, 'enabled': True, 'content_description': 'Navigate up', 'children': [], 'focused': False, 'bounds': [[0, 63], [147, 210]], 'resource_id': None, 'checked': False, 'text': None, 'class': 'android.widget.ImageButton', 'scrollable': False, 'selected': False, 'long_clickable': False, 'parent': 6, 'temp_id': 7, 'size': '147*147'}
view: {'package': 'com.gianlu.dnshero', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[189, 101], [828, 172]], 'resource_id': None, 'checked': False, 'text': 'www.google.com - DNSHero', 'class': 'android.widget.TextView', 'scrollable': False, 'selected': False, 

...

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
view: {'package': 'livio.rssreader', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[42, 94], [126, 178]], 'resource_id': None, 'checked': False, 'text': None, 'class': 'android.widget.ImageView', 'scrollable': False, 'selected': False, 'long_clickable': False, 'parent': 6, 'temp_id': 7, 'size': '84*84'}
view: {'package': 'livio.rssreader', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[126, 73], [531, 144]], 'resource_id': None, 'checked': False, 'text': 'Business Insider', 'class': 'android.widget.TextView', 'scrollable': False, 'selected': False, 'long_clickable': False, 

Traceback (most recent call last):
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/blocks.py", line 2098, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspac

APK selecionado: t20kdc.offlinepuzzlesolver_4.apk


Traceback (most recent call last):
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspaces-doutorado/workspace-rv/rvsec/rv-android/venv_rvandroid/lib/python3.12/site-packages/gradio/blocks.py", line 2098, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pedro/desenvolvimento/workspaces/workspac

alterando texto do indice: 1
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
view: {'package': 't20kdc.offlinepuzzlesolver', 'visible': True, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[42, 101], [518, 172]], 'resource_id': None, 'checked': False, 'text': 'NetGuard Challenge', 'class': 'android.widget.TextView', 'scrollable': False, 'selected': False, 'long_clickable': False, 'parent': 3, 'temp_id': 4, 'size': '476*71'}
view: {'package': 't20kdc.offlinepuzzlesolver', 'visible': False, 'checkable': False, 'child_count': 0, 'editable': False, 'clickable': False, 'is_password': False, 'focusable': False, 'enabled': True, 'content_description': None, 'children': [], 'focused': False, 'bounds': [[1080, 63], [1080, 210]], 'resource_id': None, 'checked': False, 'text': None, 'class': 'android.widget.LinearLayout', 'scr

In [ ]:
if llm:
    llm.clean()
del llm

## Summarizer

In [ ]:
summarizer_system_prompt = """You are an expert in summarizing text that will serve as input for Android application screen testers. The texts contain general information about the application and you must create a summary with data that is useful for testing the software. Identifying the activities, the transitions between them and the events that activate these transitions."""
summarizer_user_prompt = "Please summarize the following texts and break it down into smaller sections.\n{}"


In [ ]:
# Pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_text(text):
  summary = summarizer(text, max_length=500, min_length=50, do_sample=False)
  return summary[0]["summary_text"]

In [ ]:
# text_to_summarize = f"{summarizer_system_prompt}\n{summarizer_user_prompt} \n ## Texts: \n  {response_gesda} \n {response_gator} "
text_to_summarize = f"{response_gesda}  {response_gator}"
# print(text_to_summarize)

# summary = summarize_text(text_to_summarize)
# print(summary)

In [ ]:
messages = create_messages(summarizer_system_prompt, summarizer_user_prompt, f"{response_gesda}  {response_gator}")
llm = create_llm()
response = llm.generate(messages)
print(response)
llm.clean()
del llm

In [ ]:
xxx_gesda = summarize_text(response_gesda)
print(f"gesda={xxx_gesda}")

xxx_gator = summarize_text(response_gator)
print(f"\ngator={xxx_gator}")

xxx_final = summarize_text(f"{xxx_gator}\n{xxx_gesda}")
print(f"\nfinal={xxx_final}")


In [ ]:
summarizer_model = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(summarizer_model)
print("tokenizer ...")
inputs = tokenizer(text_to_summarize, return_tensors="pt", padding=True, truncation=True, max_length=2048)
print(f"inputs ({len(inputs.input_ids)}) = {inputs}")
model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model)
outputs = model.generate(inputs.input_ids,
                         attention_mask=inputs.attention_mask, # Use attention mask to avoid padding tokens
                         max_new_tokens=200,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Response={response}")

del model, inputs, outputs, tokenizer

## Tokenizer

In [ ]:
def create_prompt(messages: list[dict[str, str]], model=DEFAULT_MODEL):
    pass

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(LLAMA, trust_remote_code=True)

text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens
tokenizer.decode(tokens)
tokenizer.batch_decode(tokens)
tokenizer.get_added_vocab()


messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

# Quantization Config - this allows us to load the model into memory and use less memory
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

# The model
model = AutoModelForCausalLM.from_pretrained(DEFAULT_MODEL, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

In [ ]:
model

In [ ]:
outputs = model.generate(inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0]))

In [ ]:
# Clean up
del inputs, outputs, model
torch.cuda.empty_cache()

In [ ]:
#

In [ ]:
# response = hf.generate(messages)
# print(response)

In [ ]:
for r in result:
    g = r["generated_text"]
    # print(g)
    for x in g:
        print(x)

xxxx

In [ ]:
import boto3
import json

bedrock = boto3.client('bedrock-runtime')

body = json.dumps({
    "prompt": "\n\nHuman: Write a short story about a robot learning to feel emotions.\n\nAssistant:",
    "max_tokens_to_sample": 1000,
    "temperature": 0.5,
    "top_p": 0.9
})

model_id = "amazon.titan-tg1-xlarge"
response = bedrock.invoke_model(
    modelId=model_id,
    accept='*/*',
    contentType='application/json',
    body=body
)

response_body = json.loads(response.get('body').read().decode('utf-8'))

print(response_body['completion'])